## Паплайн для загрузки данных, добавления истинных значений и расчета метрик

В данном тесте применяем доработанную методику тестов - помимо оценки деградации модели на ранее виденных статьях, смотрим на обобщаю способность для платежей с новыми значениями статей, этот формат будет применяться при текущем скользящем мониторинге, подробнее ниже.
Здесь тест идет по данным аналогичным дате массовой загруке (см. тест 3), так как после массовой загрузке новые метки не загружались (доработка модели + интеграция с основным сервисом), поэтому период с даты отсечки тренировочной выборким 09.10.2025 по 03.12.2025.


В данном проекте:
- загрузка платежй из БД, начиная с id платежа после последнего обучения модели (могут быть новые платежи, но календарно старые - если придет новый клиент и загрузит старые выписки - эти платежи будут полезны для оценки деградации модели по пункту 1 ниже),
- отсеивание платежей с отсутствующими дополнительными колонками(статья+проект+категория донора) - без них класификация не сработает,
- разделение платежей со статьями по имеющемуся master-словарю(блок 1) и staging-словарю с новыми статьями(блок 2),
- расчет метрик:
  
    1. - добавление истинных меток по master-словарю статей, 
       - расчет метрик прогноза - смотрим деградацию модели по старым статьям, которые были в обучении, метрика должна быть очень высокой, потому что модель работает фактически как эвристика, старые размеченные вручную статьи дают очень точный сигнал даже для новых платежей;
  
    2. - разметка нового staging-словаря (эксперт/фидбэк клиентов),
       - добавление истинных значений классификации для платежей с новыми статьями. Они могут быть близкими по значению или с опечатками относительно старых - это позволит оценить обобщающую способность модели,
       - расчет метрик по платежам с новыми статьями.  


In [ ]:
import pandas as pd
import numpy as np
import random
import os


# ⏳ прогресс-бары
from tqdm import tqdm

# загрузка моделей и функци для предобработки данных
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,roc_auc_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', 120)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [23]:
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

### Загрузим и подготовим данные для прогноза

In [ ]:
# загружаем все платежи

url_down = "https://api.lemonpie.tech/api/payments/ai"
headers = {"Authorization": "Bearer YOUR_API_TOKEN"}

#start_date = str((pd.Timestamp.today().normalize() - pd.Timedelta(days=1)).date())
#end_date = start_date #str(pd.Timestamp.today().normalize().date())
#start_date = "" #"2025-10-10"
#end_date = "" #"2025-10-20"
#start_date = "2025-09-15"
#end_date = "2025-10-15"


params = {
    "limit": 5000,
    "page": 1,
    "expenditure": "incoming"
    #"date_from": start_date,
    #"date_to": end_date  
}

all_data = []

with tqdm(desc="Загружено страниц", unit=" стр", dynamic_ncols=True) as pbar:
    while True:
        response = requests.get(url_down, headers=headers, params=params)
        if response.status_code != 200:
            print(f"❌ Ошибка загрузки данных с сервера: {response.status_code}")
            break

        result = response.json()
        page_data = result.get("data", [])
        if not page_data:
            break
        
        all_data.extend(page_data)

        params["page"] += 1
        pbar.update(1)
        
# преобразуем в таблицу (вложенные поля будут с __)
data_full = pd.json_normalize(all_data, sep="__")
print(f"ℹ️ Данные загружены с сервера. Количество записей: {len(data_full)}")
data_full.to_csv("data_download.csv", index=False)

Загружено страниц: 141 стр [17:44,  7.55s/ стр]


ℹ️ Данные загружены с сервера. Количество записей: 700052


In [ ]:
# отсеиваем платежи, участвовавшие в последнем обучении, по id
train_history = pd.read_csv('train_history.csv')
last_train_payment_id = train_history['last_payment_id'].iloc[-1]

display(last_train_payment_id)

data_new = data_full[data_full['id']>last_train_payment_id].copy()

data_full['date'] = pd.to_datetime(data_full['date'], errors='coerce')

display(data_new.id.count(),data_new.uc__uc_id.isna().sum())

1101951

60114

13069

In [193]:
print('Проверим пропуски по основным признакам:\n',
    (data_new[['purpose','articles__name','projects__name','counterparties__name']].isna().sum()))
print(
    "Количество строк, где все 3 дополнительных признака отсутствуют:",
    data_new[['articles__name','projects__name','counterparties__name']]
        .isna()
        .all(axis=1)
        .sum()
)

Проверим пропуски по основным признакам:
 purpose                  213
articles__name          6212
projects__name          8621
counterparties__name    7221
dtype: int64
Количество строк, где все 3 дополнительных признака отсутствуют: 5704


In [ ]:
data_new = data_new.dropna(subset=['articles__name', 'projects__name', 'counterparties__name'], how='all') # здесь нет прогнозов изза отсутствия основных признаков
data_new = data_new.dropna(subset=['uc__uc_id']) # здесь нет прогнозов каким-то причинам - метки не загружались в связи сдоработкой модели
data_new = data_new.dropna(subset=['articles__name']) #сбраываем строки где нет значения статьи, мы не сможем истинные значения расставаить по этим строкам в соответствии со словарем
data_new.id.count()

46532

Разделим данные в соответствии со значениями поля статья по master-словарю (который использовался при обучении)

In [216]:
main_dict = pd.read_csv('main_dict.csv')

In [217]:
display(main_dict.raw.count(),data_full.articles__name.nunique())

749

835

In [ ]:
data_new['articles__name'] = data_new['articles__name'].str.lower()

# добавляем истинные метки новым платежам по основному словарю, которы участвовали в разметке тренировочной выборки
data_new_main_articles = data_new.merge(
    main_dict[['raw', 'uc_id']],
    left_on='articles__name',
    right_on='raw',
    how='left'
)

data_new_main_articles.rename(columns={'uc_id': 'target'}, inplace=True)
data_new_main_articles.drop(columns=['raw'], inplace=True)
data_new_main_articles.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id,target
0,1102113,3571,46,2025-10-08,3000.0,Пополнение по операции СБП 7159181898. Терминал khabdety,44023,incoming,4164,0,0,10283,11000,3571.0,1010.0,44023.0,1010.0,44006.0,тбанк сбп,4164.0,1010.0,4163.0,_Общие поступления на УД,11000.0,1010.0,10996.0,массовые,10283.0,1010.0,NaN,1.0,1.0
1,1102114,3571,46,2025-10-08,30.0,Пополнение по операции СБП 7159183075. Терминал khabdety,44023,incoming,4164,0,0,10283,11000,3571.0,1010.0,44023.0,1010.0,44006.0,тбанк сбп,4164.0,1010.0,4163.0,_Общие поступления на УД,11000.0,1010.0,10996.0,массовые,10283.0,1010.0,NaN,1.0,1.0
2,1102115,3571,46,2025-10-08,30.0,Пополнение по операции СБП 7159188475. Терминал khabdety,44023,incoming,4164,0,0,10283,11000,3571.0,1010.0,44023.0,1010.0,44006.0,тбанк сбп,4164.0,1010.0,4163.0,_Общие поступления на УД,11000.0,1010.0,10996.0,массовые,10283.0,1010.0,NaN,1.0,1.0


In [ ]:
# отдельно созраняем платеди с отсутстивующими истинными метками - для теста обобщающей способности модели
data_new_staging_articles = data_new_main_articles[data_new_main_articles.target.isna()].copy()

# сбрасывем эти платежи из основого массива с метками по старому словарю
data_new_main_articles = data_new_main_articles.dropna(subset=['target'])

display(data_new_main_articles.id.count(),data_new_staging_articles.id.count())

44811

1721

In [244]:
# считаем метрики прогнозов по статьям из основного словаря статей (который участвовал в обучении модели) 

accuracy = accuracy_score(data_new_main_articles['target'], data_new_main_articles['uc__uc_id'])
precision = precision_score(data_new_main_articles['target'], data_new_main_articles['uc__uc_id'], average='weighted')
recall = recall_score(data_new_main_articles['target'], data_new_main_articles['uc__uc_id'], average='macro')
f1 = f1_score(data_new_main_articles['target'], data_new_main_articles['uc__uc_id'], average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(data_new_main_articles['target'], data_new_main_articles['uc__uc_id']))

Accuracy: 0.9660351253040548
Precision (weighted): 0.9678199023793069
Recall (macro): 0.8981289859850532
F1-score (weighted): 0.9660360505214959

Отчет по классам:
              precision    recall  f1-score   support

         1.0       0.98      0.98      0.98     27307
         2.0       0.98      0.97      0.98     14067
         3.0       0.64      0.84      0.73       412
         4.0       0.95      0.95      0.95       735
         5.0       0.83      0.88      0.86       700
         6.0       0.84      0.62      0.71      1182
         7.0       0.68      0.99      0.80       298
         8.0       0.54      0.99      0.70        72
         9.0       0.57      0.87      0.69        38

    accuracy                           0.97     44811
   macro avg       0.78      0.90      0.82     44811
weighted avg       0.97      0.97      0.97     44811



На новых платежах со значениями статей аналогичными тренировочной выборке показывает метрики незначительно худшие относительно тестов при обучении ()

Загрузим staging-словарь с новыми значениями статей, раздадим истинные метки классификации и посчитаем метрики

In [245]:
data_new_staging_articles.articles__name.nunique()

64

In [246]:
data_new_staging_articles.articles__name.value_counts()

озон банк физлица                            553
3. сбп озон банк                             249
тпэй                                          79
пожертвования через сайт фонда                63
мос ру платформа                              61
тбанк физлица                                 56
туба физлица                                  56
совкомбанк_наш_сайт_сбп                       56
продобро платформа                            50
юмани нэк.134850.06                           45
озон селлеры                                  42
туба физлица сбп                              41
поступления через пао сбербанк                37
массовые (vk donut)                           32
втб физлица                                   32
дпд зоогостиница                              31
4. тбанк благотворительность                  28
совкомбанк эквайринг - продобро               21
вк добро физлица                              17
мкб_терминал                                  15
2. сайт_клауд       

In [247]:
staging_dict = pd.read_csv('staging_dict.csv')

In [248]:
staging_dict.head()

,raw,universal_category,uc_id
0,озон банк физлица,пожертвования от физических лиц (напрямую),1
1,3. сбп озон банк,пожертвования от физических лиц (напрямую),1
2,тпэй,пожертвования от физических лиц (напрямую),1
3,пожертвования через сайт фонда,пожертвования от физических лиц (напрямую),1
4,мос ру платформа,пожертвования через платформы,2


In [249]:
data_new_staging_articles.drop(columns=['target'], inplace=True)

data_new_staging_articles = data_new_staging_articles.merge(
    staging_dict[['raw', 'uc_id']],
    left_on='articles__name',
    right_on='raw',
    how='left'
)

data_new_staging_articles.rename(columns={'uc_id': 'target'}, inplace=True)
data_new_staging_articles.drop(columns=['raw'], inplace=True)
data_new_staging_articles.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id,target
0,1102148,2881,41747,2025-10-09,29.00,"//Приложение//кол-во распор. 1 //Перевод ср-в согл. реестра переводов за 08.10.2025 по Дог. 560/2017 от 07.12.2017, ...",48528,incoming,3801,0,0,13320,9985,2881.0,962.0,48528.0,962.0,41226.0,мкб_терминал,3801.0,962.0,0.0,Уставные деятельность,9985.0,962.0,9982.0,...массовые,13320.0,962.0,NaN,1.0,1
1,1102255,114,75,2025-10-09,2605.00,Перевод денежных средств по договору присоединения к условиям оказания услуг ИТО при осуществлении переводов денежны...,49607,incoming,1980,0,0,-1,2092,114.0,187.0,49607.0,187.0,47651.0,вк добро физлица,1980.0,187.0,0.0,Уставные цели,2092.0,187.0,13621.0,ВК добро,NaN,NaN,NaN,1.0,2
2,1102256,114,46,2025-10-09,13736.01,Перечисление денежных средств по Договору №ПД-1299 от 2024-01-30 по платежам за 2025-10-08. DS.2471242. Сумма удержа...,49853,incoming,1980,0,0,-1,12584,114.0,187.0,49853.0,187.0,47651.0,тбанк физлица,1980.0,187.0,0.0,Уставные цели,12584.0,187.0,13621.0,ТБанк,NaN,NaN,NaN,1.0,1


In [250]:
# считаем метрики прогнозов по статьям из staging-словаря статей (новые формулировки статей, которые модель не видела) 

accuracy = accuracy_score(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id'])
precision = precision_score(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id'], average='weighted')
recall = recall_score(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id'], average='macro')
f1 = f1_score(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id'], average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id']))

Accuracy: 0.7176060429982568
Precision (weighted): 0.6955023965576345
Recall (macro): 0.5011021524290071
F1-score (weighted): 0.6895889739807927

Отчет по классам:
              precision    recall  f1-score   support

           1       0.83      0.91      0.87      1267
           2       0.33      0.11      0.16       388
           3       0.07      0.50      0.12         8
           4       0.25      1.00      0.40         2
           5       0.62      0.62      0.62        34
           6       0.09      0.50      0.15        10
           7       0.22      0.88      0.35         8
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         3

    accuracy                           0.72      1721
   macro avg       0.27      0.50      0.30      1721
weighted avg       0.70      0.72      0.69      1721



/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(re